<a href="https://colab.research.google.com/github/myla48/datascience/blob/main/first.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

In [43]:
import pandas as pd

# Load the datasets
train_df = pd.read_csv('Train.csv')
test_df = pd.read_csv('Test.csv')
variable_definitions_df = pd.read_csv('VariableDefinitions.csv')
sample_submission_df = pd.read_csv('SampleSubmission.csv')

### Training Data (first 5 rows)

In [44]:
display(train_df.head())

,country,year,uniqueid,bank_account,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type
0,Kenya,2018,uniqueid_1,Yes,Rural,Yes,3,24,Female,Spouse,Married/Living together,Secondary education,Self employed
1,Kenya,2018,uniqueid_2,No,Rural,No,5,70,Female,Head of Household,Widowed,No formal education,Government Dependent
2,Kenya,2018,uniqueid_3,Yes,Urban,Yes,5,26,Male,Other relative,Single/Never Married,Vocational/Specialised training,Self employed
3,Kenya,2018,uniqueid_4,No,Rural,Yes,5,34,Female,Head of Household,Married/Living together,Primary education,Formally employed Private
4,Kenya,2018,uniqueid_5,No,Urban,No,8,26,Male,Child,Single/Never Married,Primary education,Informally employed


### Test Data (first 5 rows)

In [45]:
display(test_df.head())

,country,year,uniqueid,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type
0,Kenya,2018,uniqueid_6056,Urban,Yes,3,30,Male,Head of Household,Married/Living together,Secondary education,Formally employed Government
1,Kenya,2018,uniqueid_6060,Urban,Yes,7,51,Male,Head of Household,Married/Living together,Vocational/Specialised training,Formally employed Private
2,Kenya,2018,uniqueid_6065,Rural,No,3,77,Female,Parent,Married/Living together,No formal education,Remittance Dependent
3,Kenya,2018,uniqueid_6072,Rural,No,6,39,Female,Head of Household,Married/Living together,Primary education,Remittance Dependent
4,Kenya,2018,uniqueid_6073,Urban,No,3,16,Male,Child,Single/Never Married,Secondary education,Remittance Dependent


### Variable Definitions (first 5 rows)

In [46]:
display(variable_definitions_df.head())

,Variable Definitions,Unnamed: 1
0,country,Country interviewee is in.
1,year,Year survey was done in.
2,uniqueid,Unique identifier for each interviewee
3,location_type,"Type of location: Rural, Urban"
4,cellphone_access,"If interviewee has access to a cellphone: Yes, No"


### Sample Submission (first 5 rows)

In [47]:
display(sample_submission_df.head())

,unique_id,bank_account
0,uniqueid_1 x Kenya,0
1,uniqueid_2 x Kenya,0
2,uniqueid_3 x Kenya,0
3,uniqueid_4 x Kenya,0
4,uniqueid_5 x Kenya,0


### Training Data - Shape and Info

In [48]:
print('Shape of train_df:', train_df.shape)
display(train_df.info())

Shape of train_df: (23524, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23524 entries, 0 to 23523
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   country                 23524 non-null  object
 1   year                    23524 non-null  int64 
 2   uniqueid                23524 non-null  object
 3   bank_account            23524 non-null  object
 4   location_type           23524 non-null  object
 5   cellphone_access        23524 non-null  object
 6   household_size          23524 non-null  int64 
 7   age_of_respondent       23524 non-null  int64 
 8   gender_of_respondent    23524 non-null  object
 9   relationship_with_head  23524 non-null  object
 10  marital_status          23524 non-null  object
 11  education_level         23524 non-null  object
 12  job_type                23524 non-null  object
dtypes: int64(3), object(10)
memory usage: 2.3+ MB


None

### Data Preprocessing

In [49]:
# Identify categorical and numerical features
categorical_features = train_df.select_dtypes(include='object').columns.tolist()
numerical_features = train_df.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f'Categorical features: {categorical_features}')
print(f'Numerical features: {numerical_features}')

Categorical features: ['country', 'uniqueid', 'bank_account', 'location_type', 'cellphone_access', 'gender_of_respondent', 'relationship_with_head', 'marital_status', 'education_level', 'job_type']
Numerical features: ['year', 'household_size', 'age_of_respondent']


#### Handling Target Variable

The target variable `bank_account` is currently an object type ('Yes'/'No'). We need to convert it to a numerical format (0 or 1) for modeling.

In [50]:
# Convert 'bank_account' to numerical (0 or 1)
train_df['bank_account'] = train_df['bank_account'].map({'Yes': 1, 'No': 0})

# Remove 'bank_account' from categorical features list as it's now numerical and our target
categorical_features.remove('bank_account')

print(" 'bank_account' column converted to numerical (1 for Yes, 0 for No).")

 'bank_account' column converted to numerical (1 for Yes, 0 for No).


#### Missing Value and Rare Category Handling

Before one-hot encoding, we will handle any potential missing values and group rare categories in specific columns to reduce noise and improve model stability.

In [51]:
# 1. Check and fill missing values
print('Missing values in train_df before handling:')
display(train_df.isnull().sum())
print('\nMissing values in test_df before handling:')
display(test_df.isnull().sum())

# Fill any missing values with 'Unknown'
train_df.fillna('Unknown', inplace=True)
test_df.fillna('Unknown', inplace=True)

print('\nMissing values in train_df after handling:')
display(train_df.isnull().sum())
print('\nMissing values in test_df after handling:')
display(test_df.isnull().sum())

Missing values in train_df before handling:


,0
country,0
year,0
uniqueid,0
bank_account,0
location_type,0
cellphone_access,0
household_size,0
age_of_respondent,0
gender_of_respondent,0
relationship_with_head,0



Missing values in test_df before handling:


,0
country,0
year,0
uniqueid,0
location_type,0
cellphone_access,0
household_size,0
age_of_respondent,0
gender_of_respondent,0
relationship_with_head,0
marital_status,0



Missing values in train_df after handling:


,0
country,0
year,0
uniqueid,0
bank_account,0
location_type,0
cellphone_access,0
household_size,0
age_of_respondent,0
gender_of_respondent,0
relationship_with_head,0



Missing values in test_df after handling:


,0
country,0
year,0
uniqueid,0
location_type,0
cellphone_access,0
household_size,0
age_of_respondent,0
gender_of_respondent,0
relationship_with_head,0
marital_status,0


In [52]:
# 2. Group rare categories

def group_rare(df, col, threshold=100):
    freq = df[col].value_counts()
    rare = freq[freq < threshold].index
    df[col] = df[col].replace(rare, 'Other')
    print(f"Grouped rare categories in column '{col}'. Unique values after grouping: {df[col].nunique()}")

# Apply to specified categorical columns
for col in ['job_type', 'education_level', 'relationship_with_head']:
    group_rare(train_df, col)
    group_rare(test_df, col)

print('\nRare categories grouped successfully in specified columns.')

Grouped rare categories in column 'job_type'. Unique values after grouping: 10
Grouped rare categories in column 'job_type'. Unique values after grouping: 10
Grouped rare categories in column 'education_level'. Unique values after grouping: 6
Grouped rare categories in column 'education_level'. Unique values after grouping: 6
Grouped rare categories in column 'relationship_with_head'. Unique values after grouping: 6
Grouped rare categories in column 'relationship_with_head'. Unique values after grouping: 6

Rare categories grouped successfully in specified columns.


The `categorical_features` list should be updated to reflect any changes if the `bank_account` column was removed.

In [53]:
# Re-identify categorical features after cleaning, ensuring 'bank_account' is not included if already processed
categorical_features = train_df.select_dtypes(include='object').columns.tolist()
if 'bank_account' in categorical_features:
    categorical_features.remove('bank_account')
print(f"Updated categorical features: {categorical_features}")

Updated categorical features: ['country', 'uniqueid', 'location_type', 'cellphone_access', 'gender_of_respondent', 'relationship_with_head', 'marital_status', 'education_level', 'job_type']


#### Encoding Categorical Features

We will use one-hot encoding for the remaining categorical features. It's important to apply this transformation consistently to both training and test datasets. We'll drop the original columns after encoding to avoid multicollinearity.

In [54]:
from sklearn.preprocessing import OneHotEncoder

# 'uniqueid' is an identifier and should not be used for encoding or modeling
# 'bank_account' is the target and has already been converted to numerical (in the previous cell `2eb99f81`)

# Identify categorical features to be one-hot encoded (excluding 'uniqueid')
categorical_features_to_encode = [f for f in categorical_features if f != 'uniqueid']

# Initialize OneHotEncoder
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform on training data
ohe_train_features = ohe.fit_transform(train_df[categorical_features_to_encode])
ohe_train_df = pd.DataFrame(ohe_train_features, columns=ohe.get_feature_names_out(categorical_features_to_encode), index=train_df.index)

# Transform test data
ohe_test_features = ohe.transform(test_df[categorical_features_to_encode])
ohe_test_df = pd.DataFrame(ohe_test_features, columns=ohe.get_feature_names_out(categorical_features_to_encode), index=test_df.index)

# Identify numerical features
numerical_cols_train = train_df[numerical_features].copy()
numerical_cols_test = test_df[numerical_features].copy()

# Add the target variable to the training numerical data
target_col_train = train_df['bank_account'].copy()

# Concatenate all processed features for train_df_processed
# This includes numerical features, the target, and one-hot encoded features
train_df_processed = pd.concat([numerical_cols_train, target_col_train, ohe_train_df], axis=1)

# Concatenate all processed features for test_df_processed
# This includes numerical features and one-hot encoded features
test_df_processed = pd.concat([numerical_cols_test, ohe_test_df], axis=1)

print('Processed training data shape:', train_df_processed.shape)
print('Processed test data shape:', test_df_processed.shape)

display(train_df_processed.head())

Processed training data shape: (23524, 41)
Processed test data shape: (10086, 40)


,year,household_size,age_of_respondent,bank_account,country_Kenya,country_Rwanda,country_Tanzania,country_Uganda,location_type_Rural,location_type_Urban,...,job_type_Dont Know/Refuse to answer,job_type_Farming and Fishing,job_type_Formally employed Government,job_type_Formally employed Private,job_type_Government Dependent,job_type_Informally employed,job_type_No Income,job_type_Other Income,job_type_Remittance Dependent,job_type_Self employed
0,2018,3,24,1,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2018,5,70,0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,2018,5,26,1,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2018,5,34,0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2018,8,26,0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


#### Feature and Target Definition

Now that our data is preprocessed, we can separate the features (X) from the target variable (y).

In [55]:
# Define features (X) and target (y)
X = train_df_processed.drop(columns=['bank_account'])
y = train_df_processed['bank_account']

# Align columns between training and test sets after one-hot encoding
# This handles cases where test set might have fewer categories in some columns
train_cols = X.columns
test_cols = test_df_processed.columns

missing_in_test = set(train_cols) - set(test_cols)
for c in missing_in_test:
    test_df_processed[c] = 0

missing_in_train = set(test_cols) - set(train_cols)
for c in missing_in_train:
    X[c] = 0

# Ensure the order of columns is the same
X = X[train_cols]
test_df_processed = test_df_processed[train_cols]

print('Shape of X (features):', X.shape)
print('Shape of y (target):', y.shape)
print('Shape of test_df_processed (for prediction):', test_df_processed.shape)

Shape of X (features): (23524, 40)
Shape of y (target): (23524,)
Shape of test_df_processed (for prediction): (10086, 40)


### 1. Model Training

In [56]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Initialize the XGBoost model
xgb = XGBClassifier(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

# Train the model
xgb.fit(X, y)

print('XGBoost model trained successfully.')

XGBoost model trained successfully.


### 2. Model Evaluation

In [57]:
# Make predictions on the training data
y_pred_train = xgb.predict(X)

# Evaluate the model's performance on the training data
print('Training Accuracy:', accuracy_score(y, y_pred_train))
print('\nClassification Report (Training Data):\n', classification_report(y, y_pred_train))

Training Accuracy: 0.9038428838632886

Classification Report (Training Data):
               precision    recall  f1-score   support

           0       0.91      0.98      0.95     20212
           1       0.79      0.44      0.56      3312

    accuracy                           0.90     23524
   macro avg       0.85      0.71      0.75     23524
weighted avg       0.90      0.90      0.89     23524



### 3. Prediction and Submission

In [58]:
# Ensure test_df_processed has the same columns as X used for training
# This is crucial to avoid 'feature_names mismatch' error
test_df_processed_aligned = test_df_processed[X.columns]

# Predict probabilities on the test set
test_predictions_proba = xgb.predict_proba(test_df_processed_aligned)[:, 1]

# Predict class labels on the test set
test_predictions = xgb.predict(test_df_processed_aligned)

# Display the first few predictions
print('First 5 probability predictions:', test_predictions_proba[:5])
print('First 5 class predictions:', test_predictions[:5])

First 5 probability predictions: [0.87168896 0.8783932  0.03072002 0.0057433  0.03486865]
First 5 class predictions: [1 1 0 0 0]


In [59]:
test_predictions_proba = xgb.predict_proba(test_df_processed_aligned)[:, 1]
test_predictions = xgb.predict(test_df_processed_aligned)

print('First 5 probability predictions:', test_predictions_proba[:5])
print('First 5 class predictions:', test_predictions[:5])

First 5 probability predictions: [0.87168896 0.8783932  0.03072002 0.0057433  0.03486865]
First 5 class predictions: [1 1 0 0 0]


#### Create Submission File

In [60]:
# The uniqueid for the test set is needed for submission.
# It was intentionally excluded from feature engineering but is present in the original test_df.
# We need to recreate the 'uniqueid x country' format.

# Create 'uniqueid x country' for the test set from the original `test_df`
test_df['unique_id'] = test_df['uniqueid'] + ' x ' + test_df['country']

# Prepare the submission DataFrame
submission_df = pd.DataFrame({'unique_id': test_df['unique_id'], 'bank_account': test_predictions})

# Display the first few rows of the submission file
display(submission_df.head())

# Save the submission file to a CSV
submission_df.to_csv('Submission.csv', index=False)

print('\nSubmission file created successfully: Submission.csv')

,unique_id,bank_account
0,uniqueid_6056 x Kenya,1
1,uniqueid_6060 x Kenya,1
2,uniqueid_6065 x Kenya,0
3,uniqueid_6072 x Kenya,0
4,uniqueid_6073 x Kenya,0



Submission file created successfully: Submission.csv


#### Updating the Submission File

In [61]:
# The uniqueid for the test set is needed for submission.
# It was intentionally excluded from feature engineering but is present in the original test_df.
# We need to recreate the 'uniqueid x country' format.

# Create 'uniqueid x country' for the test set from the original `test_df`
test_df['unique_id'] = test_df['uniqueid'] + ' x ' + test_df['country']

# Prepare the submission DataFrame
submission_df = pd.DataFrame({'unique_id': test_df['unique_id'], 'bank_account': test_predictions})

# Display the first few rows of the updated submission file
display(submission_df.head())

# Save the updated submission file to a CSV (this will overwrite the previous one)
submission_df.to_csv('Submission.csv', index=False)

print('\nSubmission file updated successfully: Submission.csv')

,unique_id,bank_account
0,uniqueid_6056 x Kenya,1
1,uniqueid_6060 x Kenya,1
2,uniqueid_6065 x Kenya,0
3,uniqueid_6072 x Kenya,0
4,uniqueid_6073 x Kenya,0



Submission file updated successfully: Submission.csv


In [62]:
from google.colab import files

files.download('Submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>